# 01 — Exploratory Data Analysis
**Water-Smart Crop Planner · Tiruvallur, Tamil Nadu**

This notebook explores all five raw datasets and the merged feature table.
All cells run top-to-bottom on synthetic data; swap in real CSVs via `config.yaml`
and re-run to see real patterns.

> **Disclaimer:** Outputs below are derived from synthetic data and are for
> development purposes only. They do not represent real agricultural conditions.

In [ ]:
import sys
sys.path.insert(0, '..')

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from pathlib import Path

from src.utils.config_loader import load_config, any_synthetic, synthetic_dataset_names

cfg = load_config('../config/config.yaml')
print('District:', cfg['district']['name'], cfg['district']['state'])
print('Years:   ', cfg['study_period']['start_year'], '–', cfg['study_period']['end_year'])
print('Synthetic datasets:', synthetic_dataset_names(cfg))

## 1. Data Overview

In [ ]:
raw = Path('../data/raw')

weather    = pd.read_csv(raw/'weather.csv',    comment='#', parse_dates=['date'])
yield_df   = pd.read_csv(raw/'yield.csv',      comment='#')
prices_df  = pd.read_csv(raw/'prices.csv',     comment='#')
cw_df      = pd.read_csv(raw/'crop_water.csv', comment='#')
cost_df    = pd.read_csv(raw/'cost.csv',       comment='#')

for name, df in [('weather', weather), ('yield', yield_df), ('prices', prices_df),
                 ('crop_water', cw_df), ('cost', cost_df)]:
    print(f'\n{name}: {df.shape[0]} rows × {df.shape[1]} cols')
    display(df.head(3))

## 2. Missing Value Analysis

In [ ]:
def missing_summary(df, name):
    pct = (df.isna().sum() / len(df) * 100).round(2)
    return pct[pct > 0].rename(name)

miss = pd.concat([
    missing_summary(weather,   'weather'),
    missing_summary(yield_df,  'yield'),
    missing_summary(prices_df, 'prices'),
], axis=1).fillna(0)

print('Missing value % (columns with any missing):')
display(miss)
print('\nAll datasets have 0 % missing on synthetic data — expected.')

## 3. Rainfall Time Series

In [ ]:
# Annual total rainfall
weather['year'] = weather['date'].dt.year
annual_rain = weather.groupby('year')['PRECTOTCORR'].sum().reset_index()
annual_rain.columns = ['year', 'annual_rainfall_mm']

fig = px.bar(annual_rain, x='year', y='annual_rainfall_mm',
             title='Annual Rainfall — Tiruvallur (Synthetic)',
             labels={'annual_rainfall_mm': 'Rainfall (mm)', 'year': 'Year'},
             color_discrete_sequence=['#2563EB'])
fig.add_hline(y=annual_rain['annual_rainfall_mm'].mean(),
              line_dash='dash', line_color='red',
              annotation_text='Mean')
fig.show()

## 4. Yield Distributions per Crop

In [ ]:
fig = px.box(yield_df, x='crop', y='yield_kg_ha', color='season',
             title='Yield Distribution by Crop and Season (Synthetic)',
             labels={'yield_kg_ha': 'Yield (kg/ha)', 'crop': 'Crop'})
fig.update_layout(xaxis_tickangle=20)
fig.show()

In [ ]:
# Summary statistics per crop
yield_stats = (
    yield_df.groupby('crop')['yield_kg_ha']
    .agg(['mean','std','min','max'])
    .round(0)
)
yield_stats['cv_%'] = (yield_stats['std'] / yield_stats['mean'] * 100).round(1)
display(yield_stats)

## 5. Price Trends

In [ ]:
annual_price = (
    prices_df.groupby(['year','crop'])['price_inr_quintal']
    .mean().reset_index()
)

fig = px.line(annual_price, x='year', y='price_inr_quintal', color='crop',
              title='Annual Average Wholesale Price by Crop (Synthetic)',
              labels={'price_inr_quintal': 'Price (₹/quintal)', 'year': 'Year'})
fig.show()

## 6. Crop Water Requirements

In [ ]:
fig = px.bar(cw_df, x='crop', y='water_req_mm', color='season',
             barmode='group',
             title='Seasonal Crop Water Requirements (FAO-56 Estimates)',
             labels={'water_req_mm': 'Water Requirement (mm)', 'crop': 'Crop'})
fig.show()
display(cw_df.sort_values('water_req_mm', ascending=False))

## 7. Feature Correlation Heatmap

In [ ]:
features = pd.read_parquet('../data/processed/features.parquet')

num_cols = ['yield_kg_ha','seasonal_rainfall_mm','mean_tmax','mean_tmin',
            'heat_stress_days','dry_spell_days','price_inr_kg',
            'water_req_mm','cost_inr_ha']
corr = features[num_cols].corr().round(2)

fig = go.Figure(go.Heatmap(
    z=corr.values,
    x=corr.columns.tolist(),
    y=corr.index.tolist(),
    colorscale='RdBu',
    zmid=0,
    text=corr.values.round(2),
    texttemplate='%{text}',
    colorbar=dict(title='r')
))
fig.update_layout(title='Pearson Correlation — Numeric Features',
                  height=500, xaxis_tickangle=30)
fig.show()

## 8. Risk Score Summary

In [ ]:
from src.risk.risk_scorer import compute_risk_scores

risk = compute_risk_scores(cfg, features_path=Path('../data/processed/features.parquet'))

fig = px.scatter(risk, x='cv', y='p_fail', color='crop', symbol='season',
                 size='risk_score',
                 title='Risk Components: CV vs Dry-Year Failure Probability',
                 labels={'cv': 'Coefficient of Variation', 'p_fail': 'P(failure | dry year)'})
fig.show()

display(risk.sort_values('risk_score', ascending=False))

## 9. Model Comparison

In [ ]:
metrics_path = Path('../outputs/model_metrics.csv')
if metrics_path.exists():
    metrics = pd.read_csv(metrics_path)
    display(metrics)
    fig = px.bar(metrics, x='model', y=['MAE','RMSE'],
                 barmode='group',
                 title='Yield Model Comparison — Test Set Errors',
                 labels={'value': 'Error (kg/ha)', 'variable': 'Metric'})
    fig.show()
else:
    print('Run src/models/yield_model.py first to generate metrics.')

---
**End of EDA notebook.**  
All visualisations above are on synthetic data. Replace with real data and re-run for production insights.